# 1 — GBD ASD prevalence (OECD cohort)

## Objective

This notebook documents:

1. The definition of the analytical cohort (OECD member countries).
2. The creation of the OECD member dataset.
3. The manual download procedure from the Global Burden of Disease (GBD) portal.
4. The initial validation of dataset structure and key analytical dimensions.
5. The parameters used in the GBD data extraction.

The Global Burden of Disease (GBD) is a comprehensive epidemiological study
that provides standardized estimates of health metrics across countries and time.

It is developed by the Institute for Health Metrics and Evaluation (IHME),
an independent global health research institute based at the University of Washington.

## Methodological decision

The primary analytical cohort is restricted to OECD member countries.

Rationale:

- Higher institutional and health system comparability.
- Reduced structural heterogeneity relative to global groupings.
- Clear and externally defined membership criteria.

## OECD member countries source

The official list of OECD member countries is published by the OECD:

https://www.oecd.org/about/members-and-partners/

Several programmatic extraction approaches were evaluated (web scraping, OECD SDMX API, and knowledge graph queries). However:

- OECD web pages are protected by Cloudflare, preventing automated scraping.
- The OECD SDMX API provides country code lists but **does not encode OECD membership**.
- External knowledge graphs (e.g., Wikidata, DBpedia) require prior knowledge of specific data model properties.

Given that the OECD membership list is **short (38 countries), stable, and publicly accessible**, the most robust and transparent approach is to:

1. Verify the list directly from the OECD website.
2. Store it as a versioned dataset within the repository.

The canonical list used in this project is stored as:

`data/3_processed/oecd_members.csv`

## 1.1 Environment and project setup

Prepare the notebook environment and project configuration.

Includes:
- core library imports
- environment validation
- project path setup

In [1]:
# Import core libraries and project paths for data access and processing

import pandas as pd
import sys
from src.paths import RAW_DIR, PROCESSED_DIR, ensure_data_dirs

In [2]:
# Validate environment configuration and library availability

print("Python version:", sys.version.split()[0])

try:
    import src
    print("Project package import: OK")
except ImportError:
    print("ERROR: project package 'src' not found")

print("pandas version:", pd.__version__)

Python version: 3.13.5
Project package import: OK
pandas version: 2.2.3


In [3]:
# Ensure project data directories exist and validate path configuration

ensure_data_dirs()

print("RAW_DIR:", RAW_DIR)
print("PROCESSED_DIR:", PROCESSED_DIR)

RAW_DIR: C:\Users\samai\OneDrive\Escritorio\PROYECTO AUTISMO\autism-diagnosis-gender-gap\data\1_raw
PROCESSED_DIR: C:\Users\samai\OneDrive\Escritorio\PROYECTO AUTISMO\autism-diagnosis-gender-gap\data\3_processed


## 1.2 Data ingestion and validation

Load and validate the datasets used in the analysis.

Steps included:
- creation of the OECD member countries dataset
- loading of the raw GBD ASD prevalence dataset
- validation of dataset structure and key analytical dimensions

In [4]:
# Create and store canonical OECD country list used to define the analysis cohort

oecd_members_list = [
    "Australia","Austria","Belgium","Canada","Chile","Colombia","Costa Rica",
    "Czechia","Denmark","Estonia","Finland","France","Germany","Greece",
    "Hungary","Iceland","Ireland","Israel","Italy","Japan","South Korea",
    "Latvia","Lithuania","Luxembourg","Mexico","Netherlands","New Zealand",
    "Norway","Poland","Portugal","Slovakia","Slovenia","Spain","Sweden",
    "Switzerland","Turkey","United Kingdom","United States"
]

oecd_members = pd.DataFrame({"country": oecd_members_list})
oecd_members.to_csv(PROCESSED_DIR / "oecd_members.csv", index=False)

print("Saved:", PROCESSED_DIR / "oecd_members.csv")

Saved: C:\Users\samai\OneDrive\Escritorio\PROYECTO AUTISMO\autism-diagnosis-gender-gap\data\3_processed\oecd_members.csv


In [5]:
# Load OECD members dataset and validate structure and uniqueness of country list

oecd_members = pd.read_csv(PROCESSED_DIR / "oecd_members.csv")

print("Shape:", oecd_members.shape)

print("\nMissing values:")
print(oecd_members.isnull().sum())

print("\nDuplicate rows (country):")
print(oecd_members.duplicated(subset=["country"]).sum())

# Inspect OECD members sample
oecd_members.head()

Shape: (38, 1)

Missing values:
country    0
dtype: int64

Duplicate rows (country):
0


,country
0,Australia
1,Austria
2,Belgium
3,Canada
4,Chile


In [6]:
# Load raw GBD ASD prevalence dataset and validate structure and key analytical dimensions

gbd_file = RAW_DIR / "ihme_gbd_asd_prevalence_oecd_1990_2023.csv"
gbd_raw = pd.read_csv(gbd_file)

print("Shape:", gbd_raw.shape)

print("\nColumns:", list(gbd_raw.columns))

print("\nYear range:", gbd_raw["year"].min(), "-", gbd_raw["year"].max())

print("\nSex categories:", gbd_raw["sex_name"].unique())

print("\nAge categories:", gbd_raw["age_name"].unique())

print("\nMissing values (key columns):")
print(gbd_raw[["location_name", "sex_name", "age_name", "year", "val"]].isnull().sum())

print("\nDuplicate rows (full key):",
      gbd_raw.duplicated(subset=["location_name", "sex_name", "age_name", "year"]).sum())

# Inspect raw dataset sample
gbd_raw.head()


Shape: (38760, 18)

Columns: ['population_group_id', 'population_group_name', 'measure_id', 'measure_name', 'location_id', 'location_name', 'sex_id', 'sex_name', 'age_id', 'age_name', 'cause_id', 'cause_name', 'metric_id', 'metric_name', 'year', 'val', 'upper', 'lower']

Year range: 1990 - 2023

Sex categories: ['Hombre' 'Mujer']

Age categories: ['Menores de 5 años' '5-9 años' '10-14 años' '15-19 años' '20-24 años'
 '25-29 años' '30-34 años' '35-39 años' '40-44 años' '45-49 años'
 '50-54 años' '55-59 años' '60-64 años' '65-69 años' '70+ años']

Missing values (key columns):
location_name    0
sex_name         0
age_name         0
year             0
val              0
dtype: int64

Duplicate rows (full key): 0


,population_group_id,population_group_name,measure_id,measure_name,location_id,location_name,sex_id,sex_name,age_id,age_name,cause_id,cause_name,metric_id,metric_name,year,val,upper,lower
0,1,Toda la población,5,Prevalencia,55,Eslovenia,1,Hombre,1,Menores de 5 años,575,Trastornos del espectro autista,3,Tasa,1991,939.037586,1855.692963,459.595189
1,1,Toda la población,5,Prevalencia,55,Eslovenia,2,Mujer,1,Menores de 5 años,575,Trastornos del espectro autista,3,Tasa,1991,363.753608,718.021074,177.685811
2,1,Toda la población,5,Prevalencia,55,Eslovenia,1,Hombre,6,5-9 años,575,Trastornos del espectro autista,3,Tasa,1991,920.239063,1984.293942,435.700531
3,1,Toda la población,5,Prevalencia,55,Eslovenia,2,Mujer,6,5-9 años,575,Trastornos del espectro autista,3,Tasa,1991,356.527957,772.392177,168.000119
4,1,Toda la población,5,Prevalencia,55,Eslovenia,1,Hombre,7,10-14 años,575,Trastornos del espectro autista,3,Tasa,1991,912.201931,1979.073395,358.173398


## 1.3 GBD download parameters

The dataset was manually downloaded from the IHME Global Burden of Disease (GBD) Results Tool.

Source:
https://vizhub.healthdata.org/gbd-results/

The Institute for Health Metrics and Evaluation (IHME) is a global health research institute
that produces the GBD study and associated datasets.

The OECD cohort used for location selection is based on the official OECD members list:
https://www.oecd.org/about/members-and-partners/

Parameters used in the download (as provided by the GBD interface):

Measure
- Prevalencia

Metric
- Tasa

Cause
- Trastornos del espectro autista

Sex
- Hombre
- Mujer

Age groups
- Menores de 5 años
- 5-9 años
- 10-14 años
- 15-19 años
- 20-24 años
- 25-29 años
- 30-34 años
- 35-39 años
- 40-44 años
- 45-49 años
- 50-54 años
- 55-59 años
- 60-64 años
- 65-69 años
- 70+ años

Years
- 1990-2023

Locations
- 38 OECD member countries, selected manually in the GBD interface

Downloaded file stored at:

`data/1_raw/ihme_gbd_asd_prevalence_oecd_1990_2023.csv`

**Note**

Categorical values in the raw dataset are provided in Spanish, as returned by the GBD tool.
These variables are standardized to English during the ETL process in notebook 02.